# Taller POO · Scripts, terminal, `argparse` y SQLite

## Mini-iteración del proyecto: Sistema de Gestión de Biblioteca Inteligente

**Asignatura:** Programación Orientada a Objetos  
**Nivel:** primer curso de Python  
**Idea:** pasar de trabajar solo en notebooks a construir una pequeña aplicación ejecutable desde terminal.

---

## Qué vamos a construir

En este taller vamos a montar una versión **muy pequeña** del proyecto de biblioteca.

No vamos a construir todavía el proyecto final completo. Vamos a construir una **primera pieza profesional y sencilla** que luego podrá crecer:

- clases de Python para representar materiales, libros, recursos digitales, usuarios, socios y préstamos;
- una base de datos SQLite creada desde cero;
- scripts `.py` organizados en carpetas;
- una interfaz de terminal usando `argparse`;
- comandos del estilo:

```bash
python app.py add-libro --codigo L001 --titulo "El Quijote" --autor "Cervantes" --paginas 863
python app.py listar-materiales
python app.py prestar --codigo L001 --socio S001
```

La idea importante es esta:

> Hasta ahora hemos usado Python dentro de notebooks.  
> Hoy vamos a aprender a convertir una parte de nuestro código en una pequeña herramienta que otra persona pueda usar desde la terminal.

## Relación con el reto de curso

El reto de curso pide desarrollar un **Sistema de Gestión de Biblioteca Inteligente**. En ese reto aparecen materiales, usuarios, préstamos, devoluciones, reservas, búsquedas, informes, persistencia, documentación y uso desde consola.

En este taller nos centraremos solo en una parte pequeña:

| Parte del reto grande | Qué haremos hoy |
|---|---|
| Gestionar materiales | Crear y listar libros y recursos digitales |
| Gestionar usuarios | Crear y listar socios |
| Registrar préstamos/devoluciones | Prestar y devolver un material |
| Búsquedas | Buscar materiales por texto |
| Persistencia | Guardar datos en SQLite |
| Interfaz por consola | Usar `argparse` |
| Organización del código | Separar modelos, base de datos y aplicación |

No pasa nada si esta versión es limitada. Lo importante es que entendamos bien el camino.

## Objetivos de aprendizaje

Al terminar este taller deberías poder explicar:

1. Qué es un script `.py` y en qué se diferencia de un notebook.
2. Qué significa ejecutar Python desde terminal.
3. Qué es una ruta o carpeta de trabajo.
4. Qué es `argparse` y para qué sirve.
5. Cómo crear una estructura sencilla de proyecto.
6. Cómo separar clases, base de datos y comandos.
7. Cómo usar SQLite para guardar datos.
8. Cómo conectar POO + persistencia + terminal.

También deberías haber practicado:

- herencia;
- asociación;
- sobrescritura de métodos;
- polimorfismo;
- validaciones sencillas;
- consultas SQL básicas;
- lectura de argumentos por terminal.

## Normas del taller

Este notebook está pensado para ir **montando el proyecto paso a paso**.

Algunas partes están completas y otras tienen marcas como:

```python
# TODO
```

o métodos con:

```python
raise NotImplementedError("Completa este método")
```

Eso significa que esa parte es vuestra.

No se trata de copiar una solución gigante, sino de ir construyendo el puzzle:

1. Primero probamos ideas pequeñas en el notebook.
2. Después copiamos esas ideas a scripts `.py`.
3. Finalmente ejecutamos el proyecto desde terminal.

Importante:

- No vamos a usar decoradores.
- No vamos a usar librerías externas.
- Solo usaremos módulos estándar de Python: `argparse`, `sqlite3`, `pathlib` y `datetime`.
- El código estará muy comentado para que se entienda línea a línea.

# 0. Antes de empezar: notebook vs script vs terminal

## Qué hemos hecho hasta ahora

En clase hemos trabajado sobre todo con notebooks:

```text
celda 1 -> ejecutamos
celda 2 -> ejecutamos
celda 3 -> ejecutamos
```

Eso es muy cómodo para aprender, probar ideas y ver resultados intermedios.

## Qué es un script

Un script es un archivo normal de Python, por ejemplo:

```text
app.py
models.py
db.py
```

Dentro tiene código Python. La diferencia es que se ejecuta como un programa:

```bash
python app.py
```

## Qué es la terminal

La terminal es una forma de hablar con el ordenador escribiendo comandos.

En Windows puede ser:

- PowerShell
- CMD
- Terminal integrada de VS Code

En macOS o Linux puede ser:

- Terminal
- Terminal integrada de VS Code

En este taller usaremos la terminal integrada de VS Code porque es más cómoda.

## Comandos básicos de terminal

No hace falta memorizarlo todo. Usaremos muy poco.

| Qué quiero hacer | Windows PowerShell | macOS / Linux |
|---|---|---|
| Ver dónde estoy | `pwd` | `pwd` |
| Ver archivos de la carpeta | `dir` | `ls` |
| Entrar en una carpeta | `cd nombre_carpeta` | `cd nombre_carpeta` |
| Subir una carpeta | `cd ..` | `cd ..` |
| Ejecutar Python | `python archivo.py` | `python3 archivo.py` o `python archivo.py` |
| Ver versión de Python | `python --version` | `python3 --version` o `python --version` |

En Windows normalmente usaremos:

```bash
python app.py
```

En macOS/Linux quizá necesitéis:

```bash
python3 app.py
```

Durante el taller usaremos `python`. Si en vuestro ordenador no funciona, probad `python3`.

In [2]:
# Esta celda no crea todavía el proyecto.
# Solo nos ayuda a entender desde qué carpeta está trabajando el notebook.

from pathlib import Path

# Path.cwd() significa "current working directory".
# Es decir: la carpeta actual desde la que Python está trabajando.
carpeta_actual = Path.cwd()

print("La carpeta actual de este notebook es:")
print(carpeta_actual)

La carpeta actual de este notebook es:
c:\Users\elvla\OneDrive\Documentos Office\UE\2 CUATRI\PROG ORIENTADA OBJETOS\Trabajos\DIA 1\basics


# 1. Primer contacto: clases del dominio de biblioteca

Antes de crear scripts, vamos a probar las clases dentro del notebook.

Este paso es importante porque el notebook nos deja inspeccionar objetos, probar métodos y entender la lógica sin pelearnos todavía con la terminal.

## Modelo pequeño del taller

Vamos a usar estas clases:

```text
Material
├── Libro
└── RecursoDigital

Usuario
└── Socio

Prestamo
```

Relaciones importantes:

- `Libro` **es un** `Material`.
- `RecursoDigital` **es un** `Material`.
- `Socio` **es un** `Usuario`.
- `Prestamo` **asocia** un `Socio` con un `Material`.

La clase `Prestamo` no hereda de `Socio` ni de `Material`.

Un préstamo no es un socio.  
Un préstamo no es un libro.  
Un préstamo relaciona un socio y un material.

## 1.1. Clase base `Material`

La clase `Material` representa cualquier cosa que la biblioteca pueda prestar o consultar.

Tiene atributos comunes:

- código;
- título;
- autor;
- disponibilidad.

Después crearemos tipos concretos como `Libro` o `RecursoDigital`.

In [3]:
class Material:
    """
    Clase base para representar cualquier material de la biblioteca.

    En un proyecto más grande, de aquí podrían heredar:
    - Libro
    - Revista
    - RecursoDigital
    - DVD
    - etc.

    En este taller solo usaremos Libro y RecursoDigital.
    """

    def __init__(self, codigo, titulo, autor, disponible=True):
        """
        Constructor de la clase.

        Este método se ejecuta automáticamente cuando hacemos:

            material = Material(...)

        Parámetros:
        - codigo: identificador único del material. Ejemplo: "L001"
        - titulo: título del material.
        - autor: autor o responsable del material.
        - disponible: True si se puede prestar, False si ya está prestado.
        """

        # Guardamos cada dato en un atributo del objeto.
        self.codigo = codigo
        self.titulo = titulo
        self.autor = autor

        # Nos aseguramos de que disponible sea un booleano.
        self.disponible = bool(disponible)

    def prestar(self):
        """
        Intenta prestar el material.

        Si el material está disponible:
        - cambia disponible a False;
        - devuelve True.

        Si el material no está disponible:
        - no cambia nada;
        - devuelve False.
        """

        if self.disponible:
            self.disponible = False
            return True

        return False

    def devolver(self):
        """
        Marca el material como disponible otra vez.

        En esta versión simple no comprobamos quién lo tenía prestado.
        Eso lo gestionaremos más adelante con la clase Prestamo y la base de datos.
        """

        self.disponible = True

    def descripcion_corta(self):
        """
        Devuelve un texto breve para mostrar el material por pantalla.

        Este método será importante para practicar polimorfismo:
        cada subclase podrá tener su propia versión.
        """

        if self.disponible:
            estado = "disponible"
        else:
            estado = "prestado"

        return f"[{self.codigo}] {self.titulo} - {self.autor} ({estado})"

In [4]:
# Probamos la clase Material.

material = Material("M001", "Material genérico", "Autor desconocido")

print(material.descripcion_corta())

print("¿Se ha podido prestar?", material.prestar())
print(material.descripcion_corta())

print("¿Se ha podido prestar otra vez?", material.prestar())
print(material.descripcion_corta())

material.devolver()
print(material.descripcion_corta())

[M001] Material genérico - Autor desconocido (disponible)
¿Se ha podido prestar? True
[M001] Material genérico - Autor desconocido (prestado)
¿Se ha podido prestar otra vez? False
[M001] Material genérico - Autor desconocido (prestado)
[M001] Material genérico - Autor desconocido (disponible)


## 1.2. Herencia: `Libro` y `RecursoDigital`

Ahora creamos clases más concretas.

Un `Libro` es un `Material`, pero además tiene páginas.  
Un `RecursoDigital` es un `Material`, pero además tiene una URL.

Usaremos `super().__init__(...)` para reutilizar el constructor de la clase padre.

No significa nada mágico: simplemente llama al constructor de `Material` para no repetir código.

In [5]:
class Libro(Material):
    """
    Clase que representa un libro físico de la biblioteca.

    Hereda de Material porque un libro ES un material.
    """

    def __init__(self, codigo, titulo, autor, paginas, disponible=True):
        """
        Constructor de Libro.

        Recibe los datos comunes de Material y un dato propio:
        - paginas
        """

        # Llamamos al constructor de Material.
        # Así evitamos repetir:
        # self.codigo = codigo
        # self.titulo = titulo
        # self.autor = autor
        # self.disponible = disponible
        super().__init__(codigo, titulo, autor, disponible)

        # Atributo propio de Libro.
        self.paginas = int(paginas)

    def descripcion_corta(self):
        """
        TODO 1

        Completa este método.

        Debe devolver un texto parecido a:

            [L001] El Quijote - Cervantes · Libro de 863 páginas (disponible)

        Pistas:
        - puedes mirar cómo lo hace Material.descripcion_corta;
        - debes mostrar si está disponible o prestado;
        - puedes usar self.codigo, self.titulo, self.autor, self.paginas y self.disponible.
        """
        
        if self.disponible:
            estado = "disponible"
        else:
            estado = "prestado"

        return f"[{self.codigo}] {self.titulo}  - {self.autor} · Libro de {self.paginas} páginas ({estado})"


class RecursoDigital(Material):
    """
    Clase que representa un recurso digital de la biblioteca.

    Hereda de Material porque un recurso digital ES un material.
    """

    def __init__(self, codigo, titulo, autor, url, disponible=True):
        """
        Constructor de RecursoDigital.

        Recibe los datos comunes de Material y un dato propio:
        - url
        """

        super().__init__(codigo, titulo, autor, disponible)

        self.url = url

    def descripcion_corta(self):
        """
        Devuelve una descripción específica para recursos digitales.

        Este método sí está completo para que tengáis un ejemplo.
        """

        if self.disponible:
            estado = "disponible"
        else:
            estado = "prestado"

        return f"[{self.codigo}] {self.titulo} - {self.autor} · Recurso digital ({estado})"

### Ejercicio 1

Completa `Libro.descripcion_corta()` en la celda anterior y vuelve a ejecutar la celda.

Después ejecuta esta celda de prueba.

Debe imprimir dos descripciones: una de un libro y otra de un recurso digital.

In [6]:
# Esta celda fallará hasta que completes Libro.descripcion_corta().

libro = Libro("L001", "El Quijote", "Miguel de Cervantes", 863)
digital = RecursoDigital("D001", "Guía de Python", "Equipo Docente", "https://python.org")

print(libro.descripcion_corta())
print(digital.descripcion_corta())

[L001] El Quijote  - Miguel de Cervantes · Libro de 863 páginas (disponible)
[D001] Guía de Python - Equipo Docente · Recurso digital (disponible)


## 1.3. Polimorfismo

El polimorfismo aparece cuando podemos tratar objetos distintos de forma común.

Aquí tenemos una lista con materiales diferentes:

- un `Libro`;
- un `RecursoDigital`.

Ambos tienen un método llamado `descripcion_corta()`.

No nos importa de qué clase concreta sea cada objeto.  
Solo nos importa que sepa responder a ese método.

In [7]:
# Esta celda también depende de que hayas completado Libro.descripcion_corta().

materiales = [
    Libro("L002", "Clean Code", "Robert C. Martin", 464),
    RecursoDigital("D002", "Documentación de SQLite", "SQLite Team", "https://sqlite.org"),
]

for material in materiales:
    # Python decide en tiempo de ejecución qué versión del método usar.
    # Si material es Libro, usa Libro.descripcion_corta().
    # Si material es RecursoDigital, usa RecursoDigital.descripcion_corta().
    print(material.descripcion_corta())

[L002] Clean Code  - Robert C. Martin · Libro de 464 páginas (disponible)
[D002] Documentación de SQLite - SQLite Team · Recurso digital (disponible)


## 1.4. Usuarios y socios

Ahora vamos con usuarios.

En el reto grande puede haber varios perfiles:

- socio;
- bibliotecario;
- administrador.

En este taller solo implementaremos `Socio`.

La clase `Usuario` será una clase base sencilla.

In [8]:
class Usuario:
    """
    Clase base para representar usuarios del sistema.

    En este taller no la usaremos directamente.
    La usaremos como clase padre de Socio.
    """

    def __init__(self, identificador, nombre):
        """
        Constructor de Usuario.

        Parámetros:
        - identificador: código del usuario. Ejemplo: "S001"
        - nombre: nombre del usuario.
        """

        self.identificador = identificador
        self.nombre = nombre

    def descripcion_corta(self):
        """
        Devuelve una descripción sencilla del usuario.
        """

        return f"[{self.identificador}] {self.nombre}"


class Socio(Usuario):
    """
    Clase que representa un socio de la biblioteca.

    Hereda de Usuario porque un socio ES un usuario.
    """

    def __init__(self, identificador, nombre, sancionado=False, max_prestamos=3):
        """
        Constructor de Socio.

        Además de identificador y nombre, un socio tiene:
        - sancionado: indica si puede o no pedir préstamos;
        - max_prestamos: número máximo de préstamos activos permitidos.
        """

        super().__init__(identificador, nombre)

        self.sancionado = bool(sancionado)
        self.max_prestamos = int(max_prestamos)

    def puede_prestar(self, numero_prestamos_activos):
        """
        TODO 2

        Completa este método.

        Debe devolver True si el socio puede pedir un préstamo.

        Reglas:
        - Si el socio está sancionado, no puede prestar.
        - Si ya tiene max_prestamos o más préstamos activos, no puede prestar.
        - En caso contrario, sí puede prestar.

        Ejemplos:
        - Socio no sancionado con 0 préstamos activos y máximo 3 -> True
        - Socio sancionado con 0 préstamos activos -> False
        - Socio no sancionado con 3 préstamos activos y máximo 3 -> False
        """
        if numero_prestamos_activos >= 3 or self.sancionado == True:
            return False
        else:
            return True

        raise NotImplementedError("Completa Socio.puede_prestar()")

    def descripcion_corta(self):
        """
        Sobrescribimos la descripción para indicar si el socio está sancionado.
        """

        if self.sancionado:
            estado = "sancionado"
        else:
            estado = "activo"

        return f"[{self.identificador}] {self.nombre} · Socio {estado}"

### Ejercicio 2

Completa `Socio.puede_prestar()`.

Después ejecuta esta celda. Todas las comprobaciones deberían pasar.

In [9]:
# Esta celda fallará hasta que completes Socio.puede_prestar().

socio_activo = Socio("S001", "Ana García")
socio_sancionado = Socio("S002", "Luis Pérez", sancionado=True)

print(socio_activo.descripcion_corta())
print(socio_sancionado.descripcion_corta())

assert socio_activo.puede_prestar(0) == True
assert socio_activo.puede_prestar(2) == True
assert socio_activo.puede_prestar(3) == False
assert socio_sancionado.puede_prestar(0) == False

print("Pruebas superadas.")

[S001] Ana García · Socio activo
[S002] Luis Pérez · Socio sancionado
Pruebas superadas.


## 1.5. Asociación: `Prestamo`

Un préstamo une un socio y un material.

Esta relación es una **asociación**:

- un préstamo conoce al socio;
- un préstamo conoce el material;
- pero un préstamo no es un socio ni un material.

En esta primera versión la clase será muy sencilla.

In [10]:
class Prestamo:
    """
    Clase que representa el préstamo de un material a un socio.

    Esta clase muestra una asociación:
    - contiene una referencia a un Socio;
    - contiene una referencia a un Material.
    """

    def __init__(self, id_prestamo, socio, material, fecha_prestamo, fecha_devolucion=None):
        """
        Constructor de Prestamo.

        Parámetros:
        - id_prestamo: identificador del préstamo.
        - socio: objeto de tipo Socio.
        - material: objeto de tipo Material, Libro o RecursoDigital.
        - fecha_prestamo: texto con la fecha del préstamo.
        - fecha_devolucion: texto con la fecha de devolución o None si sigue activo.
        """

        self.id_prestamo = id_prestamo
        self.socio = socio
        self.material = material
        self.fecha_prestamo = fecha_prestamo
        self.fecha_devolucion = fecha_devolucion

    def esta_activo(self):
        """
        Un préstamo está activo si todavía no tiene fecha de devolución.
        """

        return self.fecha_devolucion is None

    def resumen(self):
        """
        Devuelve una frase legible para mostrar el préstamo.
        """

        if self.esta_activo():
            estado = "activo"
        else:
            estado = f"devuelto el {self.fecha_devolucion}"

        return f"Préstamo {self.id_prestamo}: {self.socio.nombre} tiene '{self.material.titulo}' ({estado})"

In [11]:
# Esta celda depende de que las clases anteriores estén completadas.

socio = Socio("S001", "Ana García")
libro = Libro("L001", "El Quijote", "Miguel de Cervantes", 863)

prestamo = Prestamo(
    id_prestamo=1,
    socio=socio,
    material=libro,
    fecha_prestamo="2026-04-30"
)

print(prestamo.resumen())

Préstamo 1: Ana García tiene 'El Quijote' (activo)


# 2. SQLite desde cero, sin scripts todavía

Antes de meter SQLite en nuestro proyecto, vamos a probarlo en pequeño.

SQLite es una base de datos que se guarda en un archivo.

No necesitamos instalar un servidor. Python ya trae el módulo `sqlite3`.

La idea básica es:

1. Abrir conexión.
2. Crear tablas si no existen.
3. Insertar datos.
4. Consultar datos.
5. Cerrar conexión.

En este bloque usaremos una base de datos de prueba llamada:

```text
demo_biblioteca.db
```

Esta base de datos no es la definitiva del proyecto. Es solo para practicar.

In [12]:
# Importamos sqlite3, que viene incluido con Python.
import sqlite3

# Path nos ayuda a trabajar con rutas de archivos y carpetas.
from pathlib import Path

# Definimos el archivo de base de datos de prueba.
ruta_demo = Path("demo_biblioteca.db")

# Si ya existía de una ejecución anterior, lo borramos para empezar limpios.
# Esto es solo para la demo del notebook.
# En una aplicación real no borraríamos la base de datos cada vez.
if ruta_demo.exists():
    ruta_demo.unlink()

# Abrimos una conexión con SQLite.
# Si el archivo no existe, SQLite lo crea.
conexion = sqlite3.connect(ruta_demo)

# Creamos una tabla muy sencilla de materiales.
conexion.execute(
    """
    CREATE TABLE materiales (
        codigo TEXT PRIMARY KEY,
        titulo TEXT NOT NULL,
        autor TEXT NOT NULL,
        disponible INTEGER NOT NULL
    )
    """
)

# Guardamos los cambios.
conexion.commit()

# Cerramos la conexión.
conexion.close()

print("Base de datos creada en:", ruta_demo)

Base de datos creada en: demo_biblioteca.db


## 2.1. Insertar datos

En SQLite usamos `INSERT INTO` para guardar datos.

Fíjate en los signos `?`.

```python
VALUES (?, ?, ?, ?)
```

Eso significa:

> Aquí irá un valor que le pasaremos desde Python.

Es mejor que construir SQL juntando textos a mano.

In [13]:
conexion = sqlite3.connect(ruta_demo)

conexion.execute(
    """
    INSERT INTO materiales (codigo, titulo, autor, disponible)
    VALUES (?, ?, ?, ?)
    """,
    ("L001", "El Quijote", "Miguel de Cervantes", 1)
)

conexion.execute(
    """
    INSERT INTO materiales (codigo, titulo, autor, disponible)
    VALUES (?, ?, ?, ?)
    """,
    ("L002", "Clean Code", "Robert C. Martin", 1)
)

conexion.commit()
conexion.close()

print("Materiales insertados.")

Materiales insertados.


## 2.2. Consultar datos

Usamos `SELECT` para leer datos.

`fetchall()` devuelve todas las filas encontradas.

In [14]:
conexion = sqlite3.connect(ruta_demo)

filas = conexion.execute(
    """
    SELECT codigo, titulo, autor, disponible
    FROM materiales
    """
).fetchall()

conexion.close()

print("Filas encontradas:")
for fila in filas:
    print(fila)

Filas encontradas:
('L001', 'El Quijote', 'Miguel de Cervantes', 1)
('L002', 'Clean Code', 'Robert C. Martin', 1)


## 2.3. Leer filas como diccionarios

Por defecto SQLite devuelve tuplas.

Podemos pedirle que devuelva filas más cómodas usando:

```python
conexion.row_factory = sqlite3.Row
```

Así podremos escribir:

```python
fila["titulo"]
```

en lugar de:

```python
fila[1]
```

In [16]:
conexion = sqlite3.connect(ruta_demo)

# Esta línea hace que las filas se comporten de forma parecida a un diccionario.
conexion.row_factory = sqlite3.Row

filas = conexion.execute(
    """
    SELECT codigo, titulo, autor, disponible
    FROM materiales
    """
).fetchall()

conexion.close()

for fila in filas:
    print(fila["codigo"], "-", fila["titulo"], "-", fila["autor"], "-", fila["disponible"])

L001 - El Quijote - Miguel de Cervantes - 1
L002 - Clean Code - Robert C. Martin - 1


## 2.4. Ejercicio rápido de SQL

Modifica la siguiente consulta para que busque materiales cuyo título contenga la palabra `"Code"`.

Pista:

```sql
WHERE titulo LIKE ?
```

y desde Python puedes pasar:

```python
("%Code%",)
```

El símbolo `%` significa “cualquier texto antes o después”.

In [18]:
conexion = sqlite3.connect(ruta_demo)
conexion.row_factory = sqlite3.Row

# TODO 3
# Cambia esta consulta para filtrar por título.
filas = conexion.execute(
    """
    SELECT codigo, titulo, autor, disponible
    FROM materiales
    WHERE titulo LIKE ?
    """,
    ("%Code",)
).fetchall()

conexion.close()

for fila in filas:
    print(fila["codigo"], fila["titulo"])

L002 Clean Code


# 3. `argparse` en pequeño, sin scripts todavía

`argparse` sirve para que un programa de Python lea argumentos escritos en la terminal.

Por ejemplo, si escribimos:

```bash
python app.py saludar --nombre Ana
```

queremos que Python entienda:

- comando: `saludar`;
- nombre: `"Ana"`.

En un script real los argumentos vendrán de la terminal.

En un notebook no tenemos terminal igual que en un script, pero podemos simular los argumentos usando una lista.

In [ ]:
import argparse

# Creamos el parser.
# El parser es el objeto encargado de entender los argumentos.
parser = argparse.ArgumentParser(
    description="Ejemplo mínimo de argparse"
)

# Creamos subcomandos.
# Un subcomando es una acción concreta: saludar, listar, prestar, etc.
subparsers = parser.add_subparsers(dest="comando")

# Creamos el subcomando "saludar".
parser_saludar = subparsers.add_parser(
    "saludar",
    help="Saluda a una persona"
)

# Añadimos un argumento obligatorio llamado --nombre.
parser_saludar.add_argument(
    "--nombre",
    required=True,
    help="Nombre de la persona a saludar"
)

# En un script real argparse leería de la terminal.
# Aquí simulamos lo que el usuario escribiría.
args = parser.parse_args(["saludar", "--nombre", "Ana"])

print("Comando recibido:", args.comando)
print("Nombre recibido:", args.nombre)

if args.comando == "saludar":
    print(f"Hola, {args.nombre}")

## 3.1. Qué es `--help`

Una de las grandes ventajas de `argparse` es que genera ayuda automáticamente.

En terminal podremos escribir:

```bash
python app.py --help
```

o:

```bash
python app.py add-libro --help
```

Eso hace que el programa explique cómo se usa.

En un proyecto de clase esto es muy útil: también sirve como documentación mínima de uso.

In [ ]:
# En notebook podemos ver la ayuda así.
# No hace falta entender todos los detalles ahora.

parser.print_help()

# 4. Estructura del proyecto del taller

Ahora sí vamos a construir el mini-proyecto de scripts.

La estructura será:

```text
taller_terminal_biblioteca/
│
├── app.py
│
├── biblioteca/
│   ├── __init__.py
│   ├── models.py
│   └── db.py
│
├── data/
│   └── biblioteca.db        <- se creará al ejecutar init-db
│
├── docs/
│   └── notas_uso.md         <- opcional
│
└── assets/
    └── README_assets.md     <- opcional
```

Qué significa cada cosa:

| Archivo/carpeta | Para qué sirve |
|---|---|
| `app.py` | Punto de entrada. El usuario lo ejecuta desde terminal. |
| `biblioteca/models.py` | Clases del dominio: Material, Libro, Socio, Prestamo... |
| `biblioteca/db.py` | Funciones para crear tablas, insertar, consultar y actualizar datos. |
| `data/` | Carpeta donde se guardará la base de datos SQLite. |
| `docs/` | Documentación sencilla del taller. |
| `assets/` | Carpeta preparada para futuros recursos. Hoy casi no la usaremos. |

`__init__.py` puede estar vacío. Sirve para indicar que `biblioteca` es un paquete de Python.

## 4.1. Crear las carpetas del proyecto

Ejecuta esta celda una vez.

Creará la estructura de carpetas, pero no resolverá los ejercicios por ti.

In [ ]:
from pathlib import Path

# Nombre de la carpeta raíz del proyecto.
# Puedes cambiarlo si quieres.
PROYECTO = Path("taller_terminal_biblioteca")

# Creamos la carpeta principal.
PROYECTO.mkdir(exist_ok=True)

# Creamos subcarpetas.
(PROYECTO / "biblioteca").mkdir(exist_ok=True)
(PROYECTO / "data").mkdir(exist_ok=True)
(PROYECTO / "docs").mkdir(exist_ok=True)
(PROYECTO / "assets").mkdir(exist_ok=True)

# Creamos un archivo __init__.py vacío.
# Este archivo permite importar código desde la carpeta biblioteca.
(PROYECTO / "biblioteca" / "__init__.py").write_text("", encoding="utf-8")

print("Estructura creada en:", PROYECTO.resolve())

## 4.2. Archivo `models.py`

Ahora vamos a crear el archivo de modelos.

Este archivo contendrá las clases principales.

Algunas partes están incompletas a propósito.  
Busca las marcas `TODO`.

Después de generar el archivo, ábrelo en VS Code y completa los métodos pendientes.

In [ ]:
%%writefile taller_terminal_biblioteca/biblioteca/models.py
"""
models.py

Este archivo contiene las clases principales del dominio de biblioteca.

Dominio significa "la parte del mundo real que estamos modelando".
En nuestro caso:
- materiales;
- libros;
- recursos digitales;
- usuarios;
- socios;
- préstamos.

IMPORTANTE:
Este archivo NO sabe nada de terminal.
Este archivo NO sabe nada de SQLite.

Solo define objetos y comportamientos propios del problema.
"""


class Material:
    """
    Clase base para cualquier material de la biblioteca.

    Un Material tiene datos comunes:
    - codigo
    - titulo
    - autor
    - disponible
    """

    def __init__(self, codigo, titulo, autor, disponible=True):
        """
        Constructor de Material.

        disponible será True si el material se puede prestar.
        disponible será False si el material ya está prestado.
        """

        self.codigo = codigo
        self.titulo = titulo
        self.autor = autor
        self.disponible = bool(disponible)

    def prestar(self):
        """
        Intenta prestar el material.

        Devuelve True si se ha podido prestar.
        Devuelve False si ya estaba prestado.
        """

        if self.disponible:
            self.disponible = False
            return True

        return False

    def devolver(self):
        """
        Marca el material como disponible.
        """

        self.disponible = True

    def descripcion_corta(self):
        """
        Devuelve un texto sencillo para mostrar por pantalla.
        """

        if self.disponible:
            estado = "disponible"
        else:
            estado = "prestado"

        return f"[{self.codigo}] {self.titulo} - {self.autor} ({estado})"


class Libro(Material):
    """
    Clase que representa un libro.

    Hereda de Material porque un libro ES un material.
    """

    def __init__(self, codigo, titulo, autor, paginas, disponible=True):
        """
        Constructor de Libro.

        Usa super() para reutilizar el constructor de Material.
        """

        super().__init__(codigo, titulo, autor, disponible)
        self.paginas = int(paginas)

    def descripcion_corta(self):
        """
        TODO 1

        Completa este método para que devuelva una descripción de un libro.

        Ejemplo esperado:

        [L001] El Quijote - Miguel de Cervantes · Libro de 863 páginas (disponible)
        """

        raise NotImplementedError("Completa Libro.descripcion_corta()")


class RecursoDigital(Material):
    """
    Clase que representa un recurso digital.

    Hereda de Material porque un recurso digital ES un material.
    """

    def __init__(self, codigo, titulo, autor, url, disponible=True):
        """
        Constructor de RecursoDigital.
        """

        super().__init__(codigo, titulo, autor, disponible)
        self.url = url

    def descripcion_corta(self):
        """
        Devuelve una descripción de un recurso digital.
        """

        if self.disponible:
            estado = "disponible"
        else:
            estado = "prestado"

        return f"[{self.codigo}] {self.titulo} - {self.autor} · Recurso digital ({estado})"


class Usuario:
    """
    Clase base para usuarios del sistema.
    """

    def __init__(self, identificador, nombre):
        """
        Constructor de Usuario.
        """

        self.identificador = identificador
        self.nombre = nombre

    def descripcion_corta(self):
        """
        Devuelve una descripción sencilla.
        """

        return f"[{self.identificador}] {self.nombre}"


class Socio(Usuario):
    """
    Clase que representa un socio de la biblioteca.

    Hereda de Usuario porque un socio ES un usuario.
    """

    def __init__(self, identificador, nombre, sancionado=False, max_prestamos=3):
        """
        Constructor de Socio.
        """

        super().__init__(identificador, nombre)

        self.sancionado = bool(sancionado)
        self.max_prestamos = int(max_prestamos)

    def puede_prestar(self, numero_prestamos_activos):
        """
        TODO 2

        Completa este método.

        Reglas:
        - Si el socio está sancionado, devuelve False.
        - Si numero_prestamos_activos es mayor o igual que max_prestamos, devuelve False.
        - En caso contrario, devuelve True.
        """

        raise NotImplementedError("Completa Socio.puede_prestar()")

    def descripcion_corta(self):
        """
        Devuelve una descripción del socio.
        """

        if self.sancionado:
            estado = "sancionado"
        else:
            estado = "activo"

        return f"[{self.identificador}] {self.nombre} · Socio {estado}"


class Prestamo:
    """
    Clase que representa un préstamo.

    Esta clase muestra una asociación:
    - tiene un socio;
    - tiene un material.

    No hereda de Socio ni de Material.
    """

    def __init__(self, id_prestamo, socio, material, fecha_prestamo, fecha_devolucion=None):
        """
        Constructor de Prestamo.
        """

        self.id_prestamo = id_prestamo
        self.socio = socio
        self.material = material
        self.fecha_prestamo = fecha_prestamo
        self.fecha_devolucion = fecha_devolucion

    def esta_activo(self):
        """
        Devuelve True si el préstamo sigue activo.
        """

        return self.fecha_devolucion is None

    def resumen(self):
        """
        Devuelve una frase resumen.
        """

        if self.esta_activo():
            estado = "activo"
        else:
            estado = f"devuelto el {self.fecha_devolucion}"

        return f"Préstamo {self.id_prestamo}: {self.socio.nombre} tiene '{self.material.titulo}' ({estado})"

## 4.3. Probar `models.py` desde el notebook

Después de completar los TODO de `models.py`, ejecuta esta celda.

Si todavía no has completado los métodos, fallará. Eso es normal.

In [ ]:
# Importamos las clases desde el archivo que acabamos de crear.
from taller_terminal_biblioteca.biblioteca.models import Libro, RecursoDigital, Socio

libro = Libro("L001", "El Quijote", "Miguel de Cervantes", 863)
digital = RecursoDigital("D001", "Guía de Python", "Equipo Docente", "https://python.org")
socio = Socio("S001", "Ana García")

print(libro.descripcion_corta())
print(digital.descripcion_corta())
print(socio.descripcion_corta())

assert socio.puede_prestar(0) == True
assert socio.puede_prestar(3) == False

print("models.py parece funcionar.")

## 4.4. Archivo `db.py`

Ahora creamos el archivo que se encarga de SQLite.

Este archivo tiene funciones, no clases, porque su responsabilidad es muy concreta:

- abrir conexión;
- crear tablas;
- insertar materiales y socios;
- consultar datos;
- registrar préstamos y devoluciones.

En un proyecto más avanzado podríamos convertir esto en clases Repository, pero para este taller es mejor mantenerlo simple.

In [ ]:
%%writefile taller_terminal_biblioteca/biblioteca/db.py
"""
db.py

Este archivo contiene funciones para trabajar con SQLite.

Responsabilidad de este archivo:
- crear la base de datos;
- insertar datos;
- consultar datos;
- actualizar disponibilidad;
- registrar préstamos y devoluciones.

Este archivo NO se encarga de leer argumentos de terminal.
Eso lo hará app.py.
"""

import sqlite3
from pathlib import Path
from datetime import date

from biblioteca.models import Libro, RecursoDigital, Socio


# Ruta base del proyecto.
# __file__ es la ruta de este archivo db.py.
# parent es la carpeta que lo contiene.
# parent.parent sube a la carpeta principal del proyecto.
BASE_DIR = Path(__file__).resolve().parent.parent

# Carpeta donde guardaremos la base de datos.
DATA_DIR = BASE_DIR / "data"

# Archivo SQLite.
DB_PATH = DATA_DIR / "biblioteca.db"


def conectar():
    """
    Abre una conexión con la base de datos.

    Si la carpeta data no existe, la crea.

    Devuelve un objeto conexión.
    """

    DATA_DIR.mkdir(exist_ok=True)

    conexion = sqlite3.connect(DB_PATH)

    # Esto permite acceder a las columnas por nombre:
    # fila["titulo"] en lugar de fila[1]
    conexion.row_factory = sqlite3.Row

    return conexion


def crear_tablas():
    """
    Crea las tablas de la base de datos si no existen.

    Usamos tres tablas:
    - materiales
    - socios
    - prestamos
    """

    conexion = conectar()

    conexion.execute(
        """
        CREATE TABLE IF NOT EXISTS materiales (
            codigo TEXT PRIMARY KEY,
            tipo TEXT NOT NULL,
            titulo TEXT NOT NULL,
            autor TEXT NOT NULL,
            paginas INTEGER,
            url TEXT,
            disponible INTEGER NOT NULL
        )
        """
    )

    conexion.execute(
        """
        CREATE TABLE IF NOT EXISTS socios (
            identificador TEXT PRIMARY KEY,
            nombre TEXT NOT NULL,
            sancionado INTEGER NOT NULL,
            max_prestamos INTEGER NOT NULL
        )
        """
    )

    conexion.execute(
        """
        CREATE TABLE IF NOT EXISTS prestamos (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            codigo_material TEXT NOT NULL,
            identificador_socio TEXT NOT NULL,
            fecha_prestamo TEXT NOT NULL,
            fecha_devolucion TEXT,
            activo INTEGER NOT NULL
        )
        """
    )

    conexion.commit()
    conexion.close()


def insertar_libro(codigo, titulo, autor, paginas):
    """
    Inserta un libro en la tabla materiales.
    """

    conexion = conectar()

    conexion.execute(
        """
        INSERT INTO materiales (codigo, tipo, titulo, autor, paginas, url, disponible)
        VALUES (?, ?, ?, ?, ?, ?, ?)
        """,
        (codigo, "libro", titulo, autor, int(paginas), None, 1)
    )

    conexion.commit()
    conexion.close()


def insertar_recurso_digital(codigo, titulo, autor, url):
    """
    Inserta un recurso digital en la tabla materiales.
    """

    conexion = conectar()

    conexion.execute(
        """
        INSERT INTO materiales (codigo, tipo, titulo, autor, paginas, url, disponible)
        VALUES (?, ?, ?, ?, ?, ?, ?)
        """,
        (codigo, "digital", titulo, autor, None, url, 1)
    )

    conexion.commit()
    conexion.close()


def insertar_socio(identificador, nombre):
    """
    Inserta un socio en la tabla socios.

    Para simplificar:
    - por defecto no está sancionado;
    - por defecto puede tener hasta 3 préstamos activos.
    """

    conexion = conectar()

    conexion.execute(
        """
        INSERT INTO socios (identificador, nombre, sancionado, max_prestamos)
        VALUES (?, ?, ?, ?)
        """,
        (identificador, nombre, 0, 3)
    )

    conexion.commit()
    conexion.close()


def fila_a_material(fila):
    """
    Convierte una fila de SQLite en un objeto Python.

    Esto conecta la base de datos con nuestras clases.

    Si la fila representa un libro, devolvemos un Libro.
    Si la fila representa un recurso digital, devolvemos un RecursoDigital.
    """

    disponible = bool(fila["disponible"])

    if fila["tipo"] == "libro":
        return Libro(
            codigo=fila["codigo"],
            titulo=fila["titulo"],
            autor=fila["autor"],
            paginas=fila["paginas"],
            disponible=disponible
        )

    if fila["tipo"] == "digital":
        return RecursoDigital(
            codigo=fila["codigo"],
            titulo=fila["titulo"],
            autor=fila["autor"],
            url=fila["url"],
            disponible=disponible
        )

    # Si el tipo no es reconocido, devolvemos None.
    # En un proyecto más avanzado podríamos lanzar una excepción.
    return None


def fila_a_socio(fila):
    """
    Convierte una fila de SQLite en un objeto Socio.
    """

    return Socio(
        identificador=fila["identificador"],
        nombre=fila["nombre"],
        sancionado=bool(fila["sancionado"]),
        max_prestamos=fila["max_prestamos"]
    )


def listar_materiales():
    """
    Devuelve todos los materiales como objetos Python.
    """

    conexion = conectar()

    filas = conexion.execute(
        """
        SELECT codigo, tipo, titulo, autor, paginas, url, disponible
        FROM materiales
        ORDER BY codigo
        """
    ).fetchall()

    conexion.close()

    materiales = []

    for fila in filas:
        material = fila_a_material(fila)

        if material is not None:
            materiales.append(material)

    return materiales


def buscar_materiales(texto):
    """
    Busca materiales cuyo título o autor contenga cierto texto.

    Ejemplo:
    buscar_materiales("python")
    """

    conexion = conectar()

    patron = f"%{texto}%"

    filas = conexion.execute(
        """
        SELECT codigo, tipo, titulo, autor, paginas, url, disponible
        FROM materiales
        WHERE titulo LIKE ? OR autor LIKE ?
        ORDER BY codigo
        """,
        (patron, patron)
    ).fetchall()

    conexion.close()

    materiales = []

    for fila in filas:
        material = fila_a_material(fila)

        if material is not None:
            materiales.append(material)

    return materiales


def listar_socios():
    """
    Devuelve todos los socios como objetos Python.
    """

    conexion = conectar()

    filas = conexion.execute(
        """
        SELECT identificador, nombre, sancionado, max_prestamos
        FROM socios
        ORDER BY identificador
        """
    ).fetchall()

    conexion.close()

    socios = []

    for fila in filas:
        socios.append(fila_a_socio(fila))

    return socios


def obtener_material(codigo):
    """
    Busca un material por código.

    Devuelve un objeto Material si existe.
    Devuelve None si no existe.
    """

    conexion = conectar()

    fila = conexion.execute(
        """
        SELECT codigo, tipo, titulo, autor, paginas, url, disponible
        FROM materiales
        WHERE codigo = ?
        """,
        (codigo,)
    ).fetchone()

    conexion.close()

    if fila is None:
        return None

    return fila_a_material(fila)


def obtener_socio(identificador):
    """
    Busca un socio por identificador.

    Devuelve un objeto Socio si existe.
    Devuelve None si no existe.
    """

    conexion = conectar()

    fila = conexion.execute(
        """
        SELECT identificador, nombre, sancionado, max_prestamos
        FROM socios
        WHERE identificador = ?
        """,
        (identificador,)
    ).fetchone()

    conexion.close()

    if fila is None:
        return None

    return fila_a_socio(fila)


def contar_prestamos_activos(identificador_socio):
    """
    Cuenta cuántos préstamos activos tiene un socio.
    """

    conexion = conectar()

    fila = conexion.execute(
        """
        SELECT COUNT(*) AS total
        FROM prestamos
        WHERE identificador_socio = ? AND activo = 1
        """,
        (identificador_socio,)
    ).fetchone()

    conexion.close()

    return fila["total"]


def prestar_material(codigo_material, identificador_socio):
    """
    Intenta prestar un material a un socio.

    Devuelve dos cosas:
    - True o False según si se pudo prestar;
    - un mensaje para mostrar al usuario.
    """

    material = obtener_material(codigo_material)

    if material is None:
        return False, "No existe un material con ese código."

    if not material.disponible:
        return False, "El material no está disponible."

    socio = obtener_socio(identificador_socio)

    if socio is None:
        return False, "No existe un socio con ese identificador."

    prestamos_activos = contar_prestamos_activos(identificador_socio)

    if not socio.puede_prestar(prestamos_activos):
        return False, "El socio no puede pedir más préstamos."

    conexion = conectar()

    conexion.execute(
        """
        UPDATE materiales
        SET disponible = 0
        WHERE codigo = ?
        """,
        (codigo_material,)
    )

    conexion.execute(
        """
        INSERT INTO prestamos (
            codigo_material,
            identificador_socio,
            fecha_prestamo,
            fecha_devolucion,
            activo
        )
        VALUES (?, ?, ?, ?, ?)
        """,
        (
            codigo_material,
            identificador_socio,
            date.today().isoformat(),
            None,
            1
        )
    )

    conexion.commit()
    conexion.close()

    return True, "Préstamo registrado correctamente."


def devolver_material(codigo_material):
    """
    Devuelve un material.

    Para simplificar:
    - buscamos un préstamo activo de ese material;
    - lo marcamos como devuelto;
    - marcamos el material como disponible.
    """

    material = obtener_material(codigo_material)

    if material is None:
        return False, "No existe un material con ese código."

    if material.disponible:
        return False, "Ese material ya aparece como disponible."

    conexion = conectar()

    fila = conexion.execute(
        """
        SELECT id
        FROM prestamos
        WHERE codigo_material = ? AND activo = 1
        """,
        (codigo_material,)
    ).fetchone()

    if fila is None:
        conexion.close()
        return False, "No hay un préstamo activo para ese material."

    conexion.execute(
        """
        UPDATE prestamos
        SET activo = 0,
            fecha_devolucion = ?
        WHERE id = ?
        """,
        (date.today().isoformat(), fila["id"])
    )

    conexion.execute(
        """
        UPDATE materiales
        SET disponible = 1
        WHERE codigo = ?
        """,
        (codigo_material,)
    )

    conexion.commit()
    conexion.close()

    return True, "Devolución registrada correctamente."


def listar_prestamos_activos():
    """
    Lista los préstamos activos.

    En vez de devolver objetos Prestamo, devolvemos filas sencillas
    con la información combinada de préstamo, socio y material.

    Esto simplifica el taller.
    """

    conexion = conectar()

    filas = conexion.execute(
        """
        SELECT
            prestamos.id,
            prestamos.fecha_prestamo,
            socios.nombre AS nombre_socio,
            materiales.titulo AS titulo_material,
            materiales.codigo AS codigo_material
        FROM prestamos
        JOIN socios
            ON prestamos.identificador_socio = socios.identificador
        JOIN materiales
            ON prestamos.codigo_material = materiales.codigo
        WHERE prestamos.activo = 1
        ORDER BY prestamos.id
        """
    ).fetchall()

    conexion.close()

    return filas

## 4.5. Archivo `app.py`

Este será el archivo que el usuario ejecutará desde terminal.

Responsabilidades de `app.py`:

- leer argumentos con `argparse`;
- llamar a funciones de `db.py`;
- mostrar mensajes al usuario.

`app.py` no debería contener la lógica interna de los préstamos.  
Esa lógica está en `db.py` y en las clases de `models.py`.

In [ ]:
%%writefile taller_terminal_biblioteca/app.py
"""
app.py

Punto de entrada de la aplicación.

Este archivo se ejecuta desde terminal con comandos como:

python app.py --help
python app.py init-db
python app.py add-libro --codigo L001 --titulo "El Quijote" --autor "Cervantes" --paginas 863
python app.py listar-materiales

La responsabilidad de este archivo es:
- leer argumentos de terminal;
- llamar a las funciones adecuadas;
- mostrar resultados al usuario.
"""

import argparse

from biblioteca import db


def construir_parser():
    """
    Crea y configura el parser de argumentos.

    Un parser es el objeto que sabe interpretar lo que el usuario escribe
    en la terminal.
    """

    parser = argparse.ArgumentParser(
        description="Mini sistema de gestión de biblioteca por terminal"
    )

    # Creamos un grupo de subcomandos.
    # Cada subcomando será una acción: init-db, add-libro, listar-materiales, etc.
    subparsers = parser.add_subparsers(dest="comando")

    # ---------------------------------------------------------
    # Comando: init-db
    # ---------------------------------------------------------
    subparsers.add_parser(
        "init-db",
        help="Crea las tablas de la base de datos"
    )

    # ---------------------------------------------------------
    # Comando: add-libro
    # ---------------------------------------------------------
    parser_add_libro = subparsers.add_parser(
        "add-libro",
        help="Añade un libro a la biblioteca"
    )

    parser_add_libro.add_argument(
        "--codigo",
        required=True,
        help="Código único del libro. Ejemplo: L001"
    )

    parser_add_libro.add_argument(
        "--titulo",
        required=True,
        help="Título del libro"
    )

    parser_add_libro.add_argument(
        "--autor",
        required=True,
        help="Autor del libro"
    )

    parser_add_libro.add_argument(
        "--paginas",
        required=True,
        type=int,
        help="Número de páginas"
    )

    # ---------------------------------------------------------
    # Comando: add-digital
    # ---------------------------------------------------------
    parser_add_digital = subparsers.add_parser(
        "add-digital",
        help="Añade un recurso digital a la biblioteca"
    )

    parser_add_digital.add_argument(
        "--codigo",
        required=True,
        help="Código único del recurso. Ejemplo: D001"
    )

    parser_add_digital.add_argument(
        "--titulo",
        required=True,
        help="Título del recurso digital"
    )

    parser_add_digital.add_argument(
        "--autor",
        required=True,
        help="Autor o responsable del recurso"
    )

    parser_add_digital.add_argument(
        "--url",
        required=True,
        help="URL del recurso digital"
    )

    # ---------------------------------------------------------
    # Comando: listar-materiales
    # ---------------------------------------------------------
    subparsers.add_parser(
        "listar-materiales",
        help="Lista todos los materiales"
    )

    # ---------------------------------------------------------
    # Comando: buscar-material
    # ---------------------------------------------------------
    parser_buscar = subparsers.add_parser(
        "buscar-material",
        help="Busca materiales por título o autor"
    )

    parser_buscar.add_argument(
        "--texto",
        required=True,
        help="Texto a buscar"
    )

    # ---------------------------------------------------------
    # Comando: add-socio
    # ---------------------------------------------------------
    parser_add_socio = subparsers.add_parser(
        "add-socio",
        help="Añade un socio"
    )

    parser_add_socio.add_argument(
        "--id",
        required=True,
        help="Identificador del socio. Ejemplo: S001"
    )

    parser_add_socio.add_argument(
        "--nombre",
        required=True,
        help="Nombre del socio"
    )

    # ---------------------------------------------------------
    # Comando: listar-socios
    # ---------------------------------------------------------
    subparsers.add_parser(
        "listar-socios",
        help="Lista todos los socios"
    )

    # ---------------------------------------------------------
    # Comando: prestar
    # ---------------------------------------------------------
    parser_prestar = subparsers.add_parser(
        "prestar",
        help="Presta un material a un socio"
    )

    parser_prestar.add_argument(
        "--codigo",
        required=True,
        help="Código del material"
    )

    parser_prestar.add_argument(
        "--socio",
        required=True,
        help="Identificador del socio"
    )

    # ---------------------------------------------------------
    # Comando: devolver
    # ---------------------------------------------------------
    parser_devolver = subparsers.add_parser(
        "devolver",
        help="Devuelve un material"
    )

    parser_devolver.add_argument(
        "--codigo",
        required=True,
        help="Código del material"
    )

    # ---------------------------------------------------------
    # Comando: listar-prestamos
    # ---------------------------------------------------------
    subparsers.add_parser(
        "listar-prestamos",
        help="Lista préstamos activos"
    )

    return parser


def main():
    """
    Función principal del programa.

    Esta función:
    1. construye el parser;
    2. lee los argumentos;
    3. decide qué acción ejecutar.
    """

    parser = construir_parser()
    args = parser.parse_args()

    # Si el usuario no escribe ningún comando, mostramos la ayuda general.
    if args.comando is None:
        parser.print_help()
        return

    if args.comando == "init-db":
        db.crear_tablas()
        print("Base de datos inicializada correctamente.")

    elif args.comando == "add-libro":
        db.insertar_libro(
            codigo=args.codigo,
            titulo=args.titulo,
            autor=args.autor,
            paginas=args.paginas
        )
        print("Libro añadido correctamente.")

    elif args.comando == "add-digital":
        db.insertar_recurso_digital(
            codigo=args.codigo,
            titulo=args.titulo,
            autor=args.autor,
            url=args.url
        )
        print("Recurso digital añadido correctamente.")

    elif args.comando == "listar-materiales":
        materiales = db.listar_materiales()

        if len(materiales) == 0:
            print("No hay materiales registrados.")
        else:
            for material in materiales:
                print(material.descripcion_corta())

    elif args.comando == "buscar-material":
        materiales = db.buscar_materiales(args.texto)

        if len(materiales) == 0:
            print("No se encontraron materiales.")
        else:
            for material in materiales:
                print(material.descripcion_corta())

    elif args.comando == "add-socio":
        db.insertar_socio(
            identificador=args.id,
            nombre=args.nombre
        )
        print("Socio añadido correctamente.")

    elif args.comando == "listar-socios":
        socios = db.listar_socios()

        if len(socios) == 0:
            print("No hay socios registrados.")
        else:
            for socio in socios:
                print(socio.descripcion_corta())

    elif args.comando == "prestar":
        ok, mensaje = db.prestar_material(
            codigo_material=args.codigo,
            identificador_socio=args.socio
        )

        if ok:
            print("OK:", mensaje)
        else:
            print("ERROR:", mensaje)

    elif args.comando == "devolver":
        ok, mensaje = db.devolver_material(args.codigo)

        if ok:
            print("OK:", mensaje)
        else:
            print("ERROR:", mensaje)

    elif args.comando == "listar-prestamos":
        prestamos = db.listar_prestamos_activos()

        if len(prestamos) == 0:
            print("No hay préstamos activos.")
        else:
            for prestamo in prestamos:
                print(
                    f"#{prestamo['id']} · "
                    f"{prestamo['nombre_socio']} tiene "
                    f"'{prestamo['titulo_material']}' "
                    f"desde {prestamo['fecha_prestamo']}"
                )


# Esta condición significa:
# "Ejecuta main() solo si este archivo se está lanzando directamente".
#
# Si otro archivo importa app.py, no se ejecutará automáticamente.
if __name__ == "__main__":
    main()

## 4.6. Crear un README sencillo

El README es un archivo de texto con instrucciones de uso.

En el proyecto final, algo parecido a esto debería existir siempre.

In [ ]:
%%writefile taller_terminal_biblioteca/README.md
# Taller Terminal Biblioteca

Mini-proyecto para practicar:

- Programación Orientada a Objetos.
- Scripts `.py`.
- Terminal.
- `argparse`.
- SQLite.

## Estructura

```text
taller_terminal_biblioteca/
├── app.py
├── biblioteca/
│   ├── __init__.py
│   ├── models.py
│   └── db.py
├── data/
├── docs/
└── assets/
```

## Primer uso

Desde la carpeta `taller_terminal_biblioteca`, ejecutar:

```bash
python app.py --help
python app.py init-db
```

## Añadir datos

```bash
python app.py add-libro --codigo L001 --titulo "El Quijote" --autor "Miguel de Cervantes" --paginas 863
python app.py add-digital --codigo D001 --titulo "Guía de Python" --autor "Equipo Docente" --url "https://python.org"
python app.py add-socio --id S001 --nombre "Ana García"
```

## Consultar datos

```bash
python app.py listar-materiales
python app.py listar-socios
python app.py buscar-material --texto "Python"
```

## Préstamos

```bash
python app.py prestar --codigo L001 --socio S001
python app.py listar-prestamos
python app.py devolver --codigo L001
```

# 5. Cómo ejecutar el proyecto desde terminal

Ahora viene la parte más importante del taller.

## Paso 1: abrir la carpeta en VS Code

Abre VS Code y abre la carpeta:

```text
taller_terminal_biblioteca
```

No abras solo el archivo `app.py`.  
Abre la carpeta completa.

## Paso 2: abrir terminal integrada

En VS Code:

```text
Terminal > New Terminal
```

o en español:

```text
Terminal > Nueva terminal
```

## Paso 3: comprobar dónde estás

Escribe:

```bash
pwd
```

Deberías estar dentro de algo parecido a:

```text
.../taller_terminal_biblioteca
```

También puedes listar archivos:

Windows:

```bash
dir
```

macOS/Linux:

```bash
ls
```

Deberías ver:

```text
app.py
biblioteca
data
docs
assets
README.md
```

## Paso 4: pedir ayuda a la aplicación

```bash
python app.py --help
```

Si eso no funciona, prueba:

```bash
python3 app.py --help
```

## 5.1. Secuencia de uso recomendada

Ejecuta estos comandos en orden.

### 1. Inicializar base de datos

```bash
python app.py init-db
```

Salida esperada:

```text
Base de datos inicializada correctamente.
```

### 2. Añadir libros

```bash
python app.py add-libro --codigo L001 --titulo "El Quijote" --autor "Miguel de Cervantes" --paginas 863
python app.py add-libro --codigo L002 --titulo "Clean Code" --autor "Robert C. Martin" --paginas 464
```

### 3. Añadir recurso digital

```bash
python app.py add-digital --codigo D001 --titulo "Guía de Python" --autor "Equipo Docente" --url "https://python.org"
```

### 4. Añadir socio

```bash
python app.py add-socio --id S001 --nombre "Ana García"
```

### 5. Listar materiales

```bash
python app.py listar-materiales
```

### 6. Listar socios

```bash
python app.py listar-socios
```

### 7. Prestar material

```bash
python app.py prestar --codigo L001 --socio S001
```

### 8. Ver préstamos activos

```bash
python app.py listar-prestamos
```

### 9. Devolver material

```bash
python app.py devolver --codigo L001
```

# 6. Errores normales y cómo leerlos

Cuando empezamos con scripts, es normal que aparezcan errores.

No significa que el ordenador os odie. Significa que hay información útil.

## Error 1: no estoy en la carpeta correcta

Si ejecutas:

```bash
python app.py --help
```

y aparece algo como:

```text
can't open file 'app.py'
```

probablemente no estás dentro de la carpeta `taller_terminal_biblioteca`.

Solución:

```bash
cd ruta/a/taller_terminal_biblioteca
```

o abre la carpeta correcta en VS Code.

## Error 2: no he inicializado la base de datos

Si intentas añadir datos antes de:

```bash
python app.py init-db
```

pueden aparecer errores de tabla inexistente.

Solución:

```bash
python app.py init-db
```

## Error 3: he repetido un código

Si añades dos veces el mismo libro:

```bash
python app.py add-libro --codigo L001 ...
```

SQLite puede decir:

```text
UNIQUE constraint failed
```

Eso significa que `codigo` es clave primaria y no puede repetirse.

## Error 4: me falta completar un TODO

Si aparece:

```text
NotImplementedError
```

significa que hay un método pendiente.

Busca en `models.py`:

```python
raise NotImplementedError(...)
```

y completa ese método.

# 7. Actividades del taller

## Actividad A · Completar `Libro.descripcion_corta`

Completa el método para que devuelva una descripción de libro.

Ejemplo:

```text
[L001] El Quijote - Miguel de Cervantes · Libro de 863 páginas (disponible)
```

Cuando el libro esté prestado, debería decir:

```text
[L001] El Quijote - Miguel de Cervantes · Libro de 863 páginas (prestado)
```

## Actividad B · Completar `Socio.puede_prestar`

Reglas:

1. Si el socio está sancionado, devuelve `False`.
2. Si ya tiene demasiados préstamos activos, devuelve `False`.
3. Si no, devuelve `True`.

## Actividad C · Probar el ciclo completo

1. Inicializa la base de datos.
2. Añade dos libros.
3. Añade un socio.
4. Lista materiales.
5. Presta un libro.
6. Lista materiales otra vez.
7. Lista préstamos.
8. Devuelve el libro.
9. Lista materiales una última vez.

## Actividad D · Añadir una mejora pequeña

Elige una:

- añadir comando para listar solo materiales disponibles;
- añadir un tipo `Revista`;
- añadir un campo `categoria`;
- añadir comando para sancionar un socio;
- añadir comando para ver historial de préstamos, no solo activos.

No hagáis todas a la vez. Elegid una y hacedla bien.

# 8. Mini-guía para ampliar hacia el proyecto final

Este taller es una primera versión.

En el proyecto final podríais evolucionarlo así:

## Más clases

```text
Material
├── Libro
├── Revista
└── RecursoDigital

Usuario
├── Socio
├── Bibliotecario
└── Administrador
```

## Más tablas

```text
materiales
socios
prestamos
reservas
sanciones
```

## Más comandos

```bash
python app.py add-revista ...
python app.py reservar ...
python app.py listar-vencidos
python app.py sancionar-socio ...
python app.py informe-materiales-prestados
```

## Más reglas de negocio

- Un socio sancionado no puede pedir préstamos.
- Un material no disponible puede reservarse.
- Un socio no puede superar cierto número de préstamos.
- Un préstamo puede estar vencido.
- Las reservas pueden tener cola de espera.

## Mejor organización

Más adelante podríais separar:

```text
models/
repositories/
services/
ui/
exceptions/
```

Pero ahora no lo haremos porque el objetivo es entender bien la primera versión.

# 9. Checklist final del estudiante

Antes de dar por terminado el taller, comprueba:

- [ ] He entendido qué es un script `.py`.
- [ ] Sé abrir la terminal integrada de VS Code.
- [ ] Sé moverme a la carpeta del proyecto.
- [ ] Sé ejecutar `python app.py --help`.
- [ ] He completado los métodos TODO.
- [ ] He creado la base de datos con `init-db`.
- [ ] He añadido materiales.
- [ ] He añadido socios.
- [ ] He prestado y devuelto un material.
- [ ] He visto cómo SQLite guarda los datos entre ejecuciones.
- [ ] Sé explicar qué hace `argparse`.
- [ ] Sé explicar por qué `Prestamo` es una asociación.
- [ ] Sé explicar dónde aparece herencia.
- [ ] Sé explicar dónde aparece polimorfismo.

# 10. Preguntas de reflexión

Responde brevemente en tu cuaderno o en un archivo de notas del equipo.

1. ¿Por qué no hemos puesto todo el código dentro de `app.py`?
2. ¿Qué ventajas tiene separar `models.py` y `db.py`?
3. ¿Por qué `Libro` hereda de `Material`?
4. ¿Por qué `Prestamo` no hereda de `Material` ni de `Socio`?
5. ¿Qué problema resuelve `argparse`?
6. ¿Qué diferencia hay entre usar `input()` y usar argumentos de terminal?
7. ¿Qué datos se guardan en SQLite?
8. ¿Qué pasaría si borramos `data/biblioteca.db`?
9. ¿Qué parte de este taller podrías reutilizar en el proyecto final?
10. ¿Qué mejorarías si tuvieras una hora más?

# 11. Apéndice: chuleta rápida de comandos

Desde la carpeta `taller_terminal_biblioteca`:

```bash
python app.py --help
python app.py init-db

python app.py add-libro --codigo L001 --titulo "El Quijote" --autor "Miguel de Cervantes" --paginas 863
python app.py add-libro --codigo L002 --titulo "Clean Code" --autor "Robert C. Martin" --paginas 464
python app.py add-digital --codigo D001 --titulo "Guía de Python" --autor "Equipo Docente" --url "https://python.org"

python app.py add-socio --id S001 --nombre "Ana García"

python app.py listar-materiales
python app.py listar-socios
python app.py buscar-material --texto "Python"

python app.py prestar --codigo L001 --socio S001
python app.py listar-prestamos
python app.py devolver --codigo L001
```

Recuerda:

- Si usas macOS/Linux y `python` no funciona, prueba `python3`.
- Si el comando no encuentra `app.py`, revisa que estás en la carpeta correcta.
- Si aparece `NotImplementedError`, te falta completar un método.